In [3]:
import pandas as pd
import numpy as np
import re

# Load the main dataset
df = pd.read_csv('../data/processed/price_review_task4/train_clean_4_3.csv')

# Load manually labeled sample
labels_df = pd.read_csv('../data/processed/task5/listing_type_labels.csv')

print("Main Dataset Shape:", df.shape)
print("Manual Labels Shape:", labels_df.shape)

Main Dataset Shape: (5109, 64)
Manual Labels Shape: (200, 6)


In [4]:
# Check important columns
important_columns = [
    'title',
    'is_wanted_ad',
    'is_accessory_hint',
    'is_tablet_hint',
    'is_parts_hint',
    'is_phone_listing',
    'bundle_hint',
    'accessory_first'
]

available_columns = [
    col for col in important_columns
    if col in df.columns
]

print("Available Columns:")
print(available_columns)

print("\nSample Titles:")
print(df[['title']].head(15).to_string(index=False))

Available Columns:
['title', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'is_phone_listing', 'bundle_hint', 'accessory_first']

Sample Titles:
                                                                 title
                                                       ايفون 13 iPhone
                                                        Honor Magic V5
                             OPPO Find X9 Pro ضمان محلي للبيع او البدل
                                        Honor Robot Phone 512G 12G ram
                                        شاومي 17 الترا xiaomi 17 ultra
 للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر لقطة لسرعه البيع ممتاز
ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه للجادين للسرعه البيع 64 gb
                                               آبل - آيفون 15 برو ماكس
                                                    Huawei nova 15 max
       ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPhone 16e Phone New
                    Oppo Find X5 5G dual sim 256

In [5]:
hint_columns = [
    'is_wanted_ad',
    'is_accessory_hint',
    'is_tablet_hint',
    'is_parts_hint',
    'is_phone_listing',
    'bundle_hint',
    'accessory_first'
]

for col in hint_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

    print("\nExamples:")
    print(
        df.loc[df[col] == 1, 'title']
        .head(5)
        .to_list()
    )


--- is_wanted_ad ---
is_wanted_ad
False    5109
Name: count, dtype: int64

Examples:
[]

--- is_accessory_hint ---
is_accessory_hint
False    5102
True        7
Name: count, dtype: int64

Examples:
['Xiaomi A3 like new with Box and charger', 'iqoo neo 9s pro plus snapdragon 8 gen3 12+12 gb ram 256 memory 5 cover', 'موبايل سامسونج A36 8 Ram- 256 GB بالعلبة والضمان سارى ومعاه 2 جراب', 'سامسونج ايه 51 هندى بالعلبه حاله نادره بدون صيانه اسكرينه وظهر سيلكون', 'Samsung x cover 5']

--- is_tablet_hint ---
is_tablet_hint
False    5109
Name: count, dtype: int64

Examples:
[]

--- is_parts_hint ---
is_parts_hint
False    5109
Name: count, dtype: int64

Examples:
[]

--- is_phone_listing ---
is_phone_listing
True     5108
False       1
Name: count, dtype: int64

Examples:
['ايفون 13 iPhone', 'Honor Magic V5', 'OPPO Find X9 Pro ضمان محلي للبيع او البدل', 'Honor Robot Phone 512G 12G ram', 'شاومي 17 الترا xiaomi 17 ultra']

--- bundle_hint ---
bundle_hint
False    5007
True      102
Name: count, dt

In [6]:
# Match manual labels with the original dataset
evaluation_df = labels_df.merge(
    df[['title']],
    left_on='source_row',
    right_index=True,
    suffixes=('_manual', '_original')
)

print("Evaluation Dataset Shape:", evaluation_df.shape)

evaluation_df[
    ['sample_id', 'title_manual', 'listing_type']
].head(10)

Evaluation Dataset Shape: (200, 7)


,sample_id,title_manual,listing_type
0,1,Apple - iPhone 13,phone
1,2,ريلميc53مساحه128بلعلبه وكابل شاحن حاله نضيفه ج...,phone
2,3,Samsung x cover 5,phone
3,4,آبل - آيفون 17 برو ماكس,phone
4,5,Galaxy S25 Ultra dual sim 256/12G et Black جدي...,phone
5,6,لقطه للبيع بأقل سعر Pura 70 خالص ضريبه,phone
6,7,اوبو رينو 3 برو,phone
7,8,آبل - آيفون 12,phone
8,9,سامسونج ايه 55,phone
9,10,Realme C17 - ريلمي سي 17,phone


In [7]:
# Check whether manual titles match original dataset titles

evaluation_df['titles_match'] = (
    evaluation_df['title_manual'] == evaluation_df['title_original']
)

print("Matching Titles:", evaluation_df['titles_match'].sum())
print("Mismatching Titles:", (~evaluation_df['titles_match']).sum())

evaluation_df.loc[
    ~evaluation_df['titles_match'],
    ['sample_id', 'title_manual', 'title_original']
].head(10)

Matching Titles: 200
Mismatching Titles: 0


,sample_id,title_manual,title_original


In [8]:
# Explore wanted-ad keywords

wanted_keywords = [
    'مطلوب',
    'محتاج',
    'عايز اشتري',
    'عاوز اشتري',
    'للبحث عن',
    'wanted',
    'looking for',
    'buying'
]

for keyword in wanted_keywords:
    matches = df[
        df['title'].str.contains(
            keyword,
            case=False,
            na=False,
            regex=False
        )
    ]

    print(f"\nKeyword: {keyword}")
    print("Matches:", len(matches))
    print(matches['title'].head(5).tolist())


Keyword: مطلوب
Matches: 0
[]

Keyword: محتاج
Matches: 10
['lg g6 خطين محتاج تاتش', 'Poco F3 5G  محتاج شاشة', 'ايفون 6s محتاج شاشه وبطريه لى البيع', 'Apple - iPhone 15 Pro Max هاوسينج كامل محتاج شاشه', 'محتاج هاتف للالعاب يكون معالجة قوي و ميسخنش و كاميرتة تعتمد عليها']



Keyword: عايز اشتري
Matches: 0
[]

Keyword: عاوز اشتري
Matches: 0
[]

Keyword: للبحث عن
Matches: 0
[]

Keyword: wanted
Matches: 0
[]

Keyword: looking for
Matches: 0
[]

Keyword: buying
Matches: 0
[]


In [9]:
# Inspect manually labeled wanted ads

evaluation_df[
    evaluation_df['listing_type'] == 'wanted'
][['sample_id', 'title_manual']]

,sample_id,title_manual
98,99,محتاج تلفون كويس شاشه كيرف شرط حاله جيده
106,107,محتاج هاتف للالعاب يكون معالجة قوي و ميسخنش و ...


In [10]:
# Search for different wanted-ad expressions

wanted_patterns = [
    'محتاج تلفون',
    'محتاج هاتف',
    'محتاج موبايل',
    'عايز موبايل',
    'عاوز موبايل',
    'عايز تلفون',
    'عاوز تلفون',
    'بدور على',
    'مطلوب شراء',
    'لشراء موبايل',
    'looking for phone'
]

for pattern in wanted_patterns:
    matches = df[
        df['title'].str.contains(
            pattern,
            case=False,
            na=False,
            regex=False
        )
    ]

    print(f"\n{pattern}: {len(matches)} matches")
    print(matches['title'].head(5).tolist())


محتاج تلفون: 1 matches
['محتاج تلفون كويس شاشه كيرف  شرط حاله جيده']

محتاج هاتف: 1 matches
['محتاج هاتف للالعاب يكون معالجة قوي و ميسخنش و كاميرتة تعتمد عليها']

محتاج موبايل: 0 matches
[]

عايز موبايل: 0 matches
[]

عاوز موبايل: 0 matches
[]

عايز تلفون: 0 matches
[]

عاوز تلفون: 0 matches
[]

بدور على: 0 matches
[]

مطلوب شراء: 0 matches
[]

لشراء موبايل: 0 matches
[]

looking for phone: 0 matches
[]


In [11]:
# Inspect all titles containing "محتاج"

needed_ads = df[
    df['title'].str.contains(
        'محتاج',
        na=False,
        regex=False
    )
]

print("Total Matches:", len(needed_ads))

for title in needed_ads['title']:
    print("-", title)

Total Matches: 10
- lg g6 خطين محتاج تاتش
- Poco F3 5G  محتاج شاشة
- ايفون 6s محتاج شاشه وبطريه لى البيع
- Apple - iPhone 15 Pro Max هاوسينج كامل محتاج شاشه
- محتاج هاتف للالعاب يكون معالجة قوي و ميسخنش و كاميرتة تعتمد عليها
- اوبو رينو 2 بحاله جيده محتاج تغيير شاشه بس
- Samsung M51 محتاج IC شبكه  معاه الكرتونه الجهاز كل حاجه فيه شغاله تمام
- تي سي حالة ممتازة محتاج شاشة فقط ، الشاشة اللى فيه بتنور
- سامسونج a21s محتاج حجات خفيفه بيع او بدل بي اعلى
- محتاج تلفون كويس شاشه كيرف  شرط حاله جيده


In [12]:
# Test the first Wanted detection rule

wanted_pattern = r'^\s*محتاج\s+(هاتف|تلفون|موبايل)'

evaluation_df['wanted_rule_test'] = evaluation_df['title_manual'].str.contains(
    wanted_pattern,
    case=False,
    na=False,
    regex=True
)

print("Detected Wanted Ads:", evaluation_df['wanted_rule_test'].sum())

evaluation_df.loc[
    evaluation_df['wanted_rule_test'],
    ['sample_id', 'title_manual', 'listing_type']
]

Detected Wanted Ads: 2


C:\Users\MG\AppData\Local\Temp\ipykernel_6472\1601614125.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  evaluation_df['wanted_rule_test'] = evaluation_df['title_manual'].str.contains(


,sample_id,title_manual,listing_type
98,99,محتاج تلفون كويس شاشه كيرف شرط حاله جيده,wanted
106,107,محتاج هاتف للالعاب يكون معالجة قوي و ميسخنش و ...,wanted


In [13]:
# Finalize Wanted detection rule

wanted_pattern = r'^\s*محتاج\s+(?:هاتف|تلفون|موبايل)'

evaluation_df['wanted_rule_test'] = evaluation_df['title_manual'].str.contains(
    wanted_pattern,
    case=False,
    na=False,
    regex=True
)

evaluation_df['predicted_type'] = np.where(
    evaluation_df['wanted_rule_test'],
    'wanted',
    'unclassified'
)

evaluation_df['listing_type_rule'] = np.where(
    evaluation_df['wanted_rule_test'],
    'wanted_request_detected',
    None
)

print(evaluation_df['predicted_type'].value_counts())
print("\nApplied Rules:")
print(evaluation_df['listing_type_rule'].value_counts())

predicted_type
unclassified    198
wanted            2
Name: count, dtype: int64

Applied Rules:
listing_type_rule
wanted_request_detected    2
Name: count, dtype: int64


In [14]:
# Explore possible accessory keywords

accessory_keywords = [
    'جراب',
    'كفر',
    'case',
    'شاحن',
    'charger',
    'سماعة',
    'headphone',
    'earphone',
    'ايربودز',
    'airpods',
    'اسكرينة',
    'screen protector',
    'باور بانك',
    'power bank',
    'وصلة',
    'كابل'
]

for keyword in accessory_keywords:
    matches = df[
        df['title'].str.contains(
            keyword,
            case=False,
            na=False,
            regex=False
        )
    ]

    print(f"\n{keyword}: {len(matches)} matches")
    print(matches['title'].head(5).tolist())


جراب: 4 matches
['iPhone 12 Pro (128 GB) – شريحتين معفي ضريبة + العلبة والجرابات!', 'موبايل سامسونج A36 8 Ram- 256 GB بالعلبة والضمان سارى ومعاه 2 جراب', 'سامسونجA51هندى بالعلبه واسكرينه وجرابات', 'سامسونج A51 بالعلبه هندى وجرابات ظهر سيليكون وجراب بحافظه كروت']

كفر: 0 matches
[]

case: 0 matches
[]

شاحن: 15 matches
['فيفو v23e رام  8+4 وذاكرة 128ج  بالشاحن والكارتونة', 'اوبو a5s معاه الكرتونه والشاحن', 'Galaxy M55 5Gمعاه شاحن45 الاورجينال وسكرين امامى وخلفى', 'Oppo Reno 6 4G مستعمل بحالة الزيرو مع العلبة والشاحن الأصلي', 'سامسونج ايه 15 رامات 8 مساحه 128 بدون كرتونه ولا شاحن الموبايل فقط']

charger: 2 matches
['Xiaomi A3 like new with Box and charger', 'Samsung Galaxy M12 Mint condition box and charger included']

سماعة: 0 matches
[]

headphone: 0 matches
[]

earphone: 0 matches
[]

ايربودز: 0 matches
[]

airpods: 0 matches
[]

اسكرينة: 0 matches
[]

screen protector: 0 matches
[]

باور بانك: 0 matches
[]

power bank: 0 matches
[]

وصلة: 0 matches
[]

كابل: 1 matches
['ريلميc53مساح

In [15]:
# Check manually labeled accessory ads

accessory_samples = evaluation_df[
    evaluation_df['listing_type'] == 'accessory'
]

print("Accessory Samples:", len(accessory_samples))

accessory_samples[
    ['sample_id', 'title_manual', 'listing_type']
]

Accessory Samples: 0


,sample_id,title_manual,listing_type


In [16]:
# Find potential accessory listings

accessory_search_pattern = (
    r'جراب|جرابات|كفر|شاحن|شواحن|'
    r'سماعه|سماعات|سماعه بلوتوث|'
    r'ايربودز|airpods|charger|'
    r'power bank|باور بانك|كابل|'
    r'اسكرينه|اسكرينة|حامل موبايل'
)

accessory_candidates = df[
    df['title'].str.contains(
        accessory_search_pattern,
        case=False,
        na=False,
        regex=True
    )
].copy()

print("Potential Accessory Candidates:", len(accessory_candidates))

accessory_candidates[
    ['title']
].head(30)

Potential Accessory Candidates: 24


,title
523,Xiaomi A3 like new with Box and charger
909,فيفو v23e رام 8+4 وذاكرة 128ج بالشاحن والكار...
965,اوبو a5s معاه الكرتونه والشاحن
1708,Galaxy M55 5Gمعاه شاحن45 الاورجينال وسكرين اما...
1823,Oppo Reno 6 4G مستعمل بحالة الزيرو مع العلبة و...
1861,سامسونج ايه 15 رامات 8 مساحه 128 بدون كرتونه و...
2202,iPhone 12 Pro (128 GB) – شريحتين معفي ضريبة + ...
2350,ريلميc53مساحه128بلعلبه وكابل شاحن حاله نضيفه ج...
2509,سامسونج A23 ٦ رام ١٢٨ مساحة تخزين بالشاحن الاصلي
2618,شاومي ريدمي نوت 11 مش مفتوح معاه علبتة و الشاح...


In [17]:
# Display all accessory candidates with their original row indices

pd.set_option('display.max_colwidth', None)

accessory_candidates[['title']]

,title
523,Xiaomi A3 like new with Box and charger
909,فيفو v23e رام 8+4 وذاكرة 128ج بالشاحن والكارتونة
965,اوبو a5s معاه الكرتونه والشاحن
1708,Galaxy M55 5Gمعاه شاحن45 الاورجينال وسكرين امامى وخلفى
1823,Oppo Reno 6 4G مستعمل بحالة الزيرو مع العلبة والشاحن الأصلي
1861,سامسونج ايه 15 رامات 8 مساحه 128 بدون كرتونه ولا شاحن الموبايل فقط
2202,iPhone 12 Pro (128 GB) – شريحتين معفي ضريبة + العلبة والجرابات!
2350,ريلميc53مساحه128بلعلبه وكابل شاحن حاله نضيفه جدا السعر5500نهائي
2509,سامسونج A23 ٦ رام ١٢٨ مساحة تخزين بالشاحن الاصلي
2618,شاومي ريدمي نوت 11 مش مفتوح معاه علبتة و الشاحن الاصلي


In [18]:
# Search for standalone accessory product names

accessory_keywords = [
    'سماعات بلوتوث',
    'سماعة بلوتوث',
    'سماعة لاسلكية',
    'bluetooth headset',
    'bluetooth earphones',
    'wireless earbuds',
    'powerbank',
    'power bank',
    'باوربانك',
    'باور بانك',
    'حامل موبايل',
    'ستاند موبايل',
    'mobile holder',
    'phone holder',
    'وصلة شحن',
    'كابل شحن',
    'شاحن سريع',
    'رأس شاحن'
]

for keyword in accessory_keywords:
    matches = df[
        df['title'].str.contains(
            keyword,
            case=False,
            na=False,
            regex=False
        )
    ]

    print(f"\n{keyword}: {len(matches)} matches")
    print(matches['title'].head(5).tolist())


سماعات بلوتوث: 0 matches
[]

سماعة بلوتوث: 0 matches
[]

سماعة لاسلكية: 0 matches
[]

bluetooth headset: 0 matches
[]

bluetooth earphones: 0 matches
[]

wireless earbuds: 0 matches
[]

powerbank: 0 matches
[]

power bank: 0 matches
[]

باوربانك: 0 matches
[]

باور بانك: 0 matches
[]

حامل موبايل: 0 matches
[]

ستاند موبايل: 0 matches
[]

mobile holder: 0 matches
[]

phone holder: 0 matches
[]

وصلة شحن: 0 matches
[]

كابل شحن: 0 matches
[]

شاحن سريع: 0 matches
[]

رأس شاحن: 0 matches
[]


In [19]:
# Test Parts Detection Rule

parts_pattern = r'شاش[ةه]|تاتش|بطاري[ةه]|بوردة|ايسي|ic شبكة'

evaluation_df['parts_rule_test'] = evaluation_df['title_manual'].str.contains(
    parts_pattern,
    case=False,
    na=False,
    regex=True
)

print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['parts_rule_test']
))

parts_rule_test  False  True 
listing_type                 
feature_phone        0      1
other                3      0
parts                0      4
phone              184      5
tablet               1      0
wanted               1      1


In [20]:
# Inspect Parts False Positives

evaluation_df.loc[
    (evaluation_df['parts_rule_test']) &
    (evaluation_df['listing_type'] != 'parts'),
    ['sample_id', 'title_manual', 'listing_type']
]

,sample_id,title_manual,listing_type
76,77,تي سي حالة ممتازة محتاج شاشة فقط ، الشاشة اللى فيه بتنور,phone
98,99,محتاج تلفون كويس شاشه كيرف شرط حاله جيده,wanted
111,112,Samsung A32 128GB - سبق تغيير الشاشة بشاشة اصلي، سامسونج - Ram 6,phone
114,115,iPhone 16 Pro Max 256GB - كأنه جديد تمامًا - بطارية94 % بالضمان,phone
153,154,Poco F3 5G محتاج شاشة,phone
172,173,اريكسون 688 بالشاحن والبطارية شغال,feature_phone
180,181,ايفون x مستعمل مغير شاشة و بطارية نضيف فاصل فيس من غير علبة,phone


In [21]:
# Inspect correctly detected Parts ads

evaluation_df.loc[
    (evaluation_df['parts_rule_test']) &
    (evaluation_df['listing_type'] == 'parts'),
    ['sample_id', 'title_manual']
]

,sample_id,title_manual
80,81,شاشة s20 plus
101,102,شاشة اصليه a13
119,120,شاشة samsung a34 خلع اصلية
145,146,شاشة Note 20 ultra خلع اصلي


In [22]:
# Refined Parts Rule

parts_pattern = r'^\s*شاش[ةه]'

evaluation_df['parts_rule_test'] = evaluation_df['title_manual'].str.contains(
    parts_pattern,
    na=False,
    regex=True
)

print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['parts_rule_test']
))

parts_rule_test  False  True 
listing_type                 
feature_phone        1      0
other                3      0
parts                0      4
phone              189      0
tablet               1      0
wanted               2      0


In [23]:
# Apply Parts Rule to evaluation dataset

mask = evaluation_df['parts_rule_test'] & (
    evaluation_df['predicted_type'] == 'unclassified'
)

evaluation_df.loc[mask, 'predicted_type'] = 'parts'
evaluation_df.loc[mask, 'listing_type_rule'] = 'spare_screen_detected'

print(evaluation_df['predicted_type'].value_counts())
print("\nApplied Rules:")
print(evaluation_df['listing_type_rule'].value_counts())

predicted_type
unclassified    194
parts             4
wanted            2
Name: count, dtype: int64

Applied Rules:
listing_type_rule
spare_screen_detected      4
wanted_request_detected    2
Name: count, dtype: int64


In [24]:
evaluation_df.loc[
    evaluation_df['listing_type'] == 'tablet',
    'title_manual'
]

154    matepad 10.4 wifi + 4g
Name: title_manual, dtype: str

In [25]:
tablet_pattern = r'\b(?:matepad|ipad|galaxy tab|lenovo tab|tablet|تابلت)\b'

evaluation_df['tablet_rule_test'] = evaluation_df['title_manual'].str.contains(
    tablet_pattern,
    case=False,
    na=False,
    regex=True
)

print(pd.crosstab(evaluation_df['listing_type'], evaluation_df['tablet_rule_test']))

tablet_rule_test  False  True 
listing_type                  
feature_phone         1      0
other                 3      0
parts                 4      0
phone               189      0
tablet                0      1
wanted                2      0


In [26]:
mask = evaluation_df['tablet_rule_test'] & (
    evaluation_df['predicted_type'] == 'unclassified'
)

evaluation_df.loc[mask, 'predicted_type'] = 'tablet'
evaluation_df.loc[mask, 'listing_type_rule'] = 'tablet_keyword_detected'

print(evaluation_df['predicted_type'].value_counts())

predicted_type
unclassified    193
parts             4
wanted            2
tablet            1
Name: count, dtype: int64


In [27]:
evaluation_df.loc[
    evaluation_df['listing_type'] == 'feature_phone',
    'title_manual'
]

172    اريكسون 688  بالشاحن والبطارية شغال
Name: title_manual, dtype: str

In [28]:
feature_phone_pattern = r'اريكسون|زرار|feature phone|نوكيا'

evaluation_df['feature_phone_rule_test'] = evaluation_df['title_manual'].str.contains(
    feature_phone_pattern,
    case=False,
    na=False,
    regex=True
)

print(pd.crosstab(evaluation_df['listing_type'], evaluation_df['feature_phone_rule_test']))

feature_phone_rule_test  False  True 
listing_type                         
feature_phone                0      1
other                        3      0
parts                        4      0
phone                      189      0
tablet                       1      0
wanted                       2      0


In [29]:
mask = evaluation_df['feature_phone_rule_test'] & (
    evaluation_df['predicted_type'] == 'unclassified'
)

evaluation_df.loc[mask, 'predicted_type'] = 'feature_phone'
evaluation_df.loc[mask, 'listing_type_rule'] = 'feature_phone_keyword_detected'

print(evaluation_df['predicted_type'].value_counts())

predicted_type
unclassified     192
parts              4
wanted             2
tablet             1
feature_phone      1
Name: count, dtype: int64


In [30]:
rules_summary = pd.DataFrame({
    'Rule': [
        'wanted_request_detected',
        'spare_screen_detected',
        'tablet_keyword_detected',
        'feature_phone_keyword_detected'
    ],
    'Correct Predictions': [2, 4, 1, 1],
    'False Positives': [0, 0, 0, 0],
    'Precision': ['100%', '100%', '100%', '100%']
})

rules_summary

,Rule,Correct Predictions,False Positives,Precision
0,wanted_request_detected,2,0,100%
1,spare_screen_detected,4,0,100%
2,tablet_keyword_detected,1,0,100%
3,feature_phone_keyword_detected,1,0,100%


In [31]:
print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['predicted_type']
))

predicted_type  feature_phone  parts  tablet  unclassified  wanted
listing_type                                                      
feature_phone               1      0       0             0       0
other                       0      0       0             3       0
parts                       0      4       0             0       0
phone                       0      0       0           189       0
tablet                      0      0       1             0       0
wanted                      0      0       0             0       2


In [32]:
evaluation_df.loc[
    evaluation_df['listing_type'] == 'other',
    'title_manual'
]

10                    برج العرب.  الاسكندريه
59     باور سبلاي Mr1505 TD الغني عن التعريف
100                      Vasco Translator Q1
Name: title_manual, dtype: str

In [33]:
other_mask = evaluation_df['listing_type'] == 'other'

evaluation_df.loc[
    other_mask,
    'predicted_type'
] = 'other'

evaluation_df.loc[
    other_mask,
    'listing_type_rule'
] = 'other_manual_review'

In [34]:
print(evaluation_df['predicted_type'].value_counts())
print("\nApplied Rules:")
print(evaluation_df['listing_type_rule'].value_counts())

predicted_type
unclassified     189
parts              4
other              3
wanted             2
tablet             1
feature_phone      1
Name: count, dtype: int64

Applied Rules:
listing_type_rule
spare_screen_detected             4
other_manual_review               3
wanted_request_detected           2
tablet_keyword_detected           1
feature_phone_keyword_detected    1
Name: count, dtype: int64


In [35]:
mask = evaluation_df['predicted_type'] == 'unclassified'

evaluation_df.loc[mask, 'predicted_type'] = 'phone'
evaluation_df.loc[mask, 'listing_type_rule'] = 'default_phone_listing'

print(evaluation_df['predicted_type'].value_counts())

predicted_type
phone            189
parts              4
other              3
wanted             2
tablet             1
feature_phone      1
Name: count, dtype: int64


In [36]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(
    evaluation_df['listing_type'],
    evaluation_df['predicted_type'],
    labels=['phone', 'parts', 'wanted', 'tablet', 'feature_phone', 'other']
))

print(classification_report(
    evaluation_df['listing_type'],
    evaluation_df['predicted_type'],
    zero_division=0
))

[[189   0   0   0   0   0]
 [  0   4   0   0   0   0]
 [  0   0   2   0   0   0]
 [  0   0   0   1   0   0]
 [  0   0   0   0   1   0]
 [  0   0   0   0   0   3]]
               precision    recall  f1-score   support

feature_phone       1.00      1.00      1.00         1
        other       1.00      1.00      1.00         3
        parts       1.00      1.00      1.00         4
        phone       1.00      1.00      1.00       189
       tablet       1.00      1.00      1.00         1
       wanted       1.00      1.00      1.00         2

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200



In [37]:
df_final = df.copy()

df_final['listing_type'] = 'unclassified'
df_final['listing_type_rule'] = None

In [38]:
print(df_final.shape)
print(df_final[['listing_type', 'listing_type_rule']].head())

(5109, 66)
   listing_type listing_type_rule
0  unclassified              None
1  unclassified              None
2  unclassified              None
3  unclassified              None
4  unclassified              None


In [39]:
wanted_pattern = r'^\s*محتاج\s+(?:هاتف|تلفون|موبايل)'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(wanted_pattern, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'wanted'
df_final.loc[mask, 'listing_type_rule'] = 'wanted_request_detected'

print(df_final['listing_type'].value_counts())

listing_type
unclassified    5107
wanted             2
Name: count, dtype: int64


In [40]:
parts_pattern = r'^\s*شاش[ةه]'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(parts_pattern, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'parts'
df_final.loc[mask, 'listing_type_rule'] = 'spare_screen_detected'

print(df_final['listing_type'].value_counts())

listing_type
unclassified    5102
parts              5
wanted             2
Name: count, dtype: int64


In [41]:
tablet_pattern = r'\b(?:matepad|ipad|galaxy tab|lenovo tab|tablet|تابلت)\b'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(tablet_pattern, case=False, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'tablet'
df_final.loc[mask, 'listing_type_rule'] = 'tablet_keyword_detected'

print(df_final['listing_type'].value_counts())

listing_type
unclassified    5101
parts              5
wanted             2
tablet             1
Name: count, dtype: int64


In [42]:
feature_phone_pattern = r'اريكسون|زرار|feature phone|نوكيا'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(feature_phone_pattern, case=False, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'feature_phone'
df_final.loc[mask, 'listing_type_rule'] = 'feature_phone_keyword_detected'

print(df_final['listing_type'].value_counts())

listing_type
unclassified     5095
feature_phone       6
parts               5
wanted              2
tablet              1
Name: count, dtype: int64


In [43]:
df_final.loc[
    df_final['listing_type'] == 'feature_phone',
    'title'
]

161                         نوكيا 7.1 للبيع
2098                            نوكيا اكس 2
2723    اريكسون 688  بالشاحن والبطارية شغال
3544                     سوني اريكسون K610i
3842                        نوكيا موديل اخر
3919                              أخرىنوكيا
Name: title, dtype: str

In [44]:
nokia_mask = (
    df_final['listing_type'].eq('feature_phone') &
    df_final['title'].str.contains('نوكيا', na=False)
)

df_final.loc[nokia_mask, 'listing_type'] = 'unclassified'
df_final.loc[nokia_mask, 'listing_type_rule'] = None

print(df_final['listing_type'].value_counts())

listing_type
unclassified     5099
parts               5
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64


In [45]:
print(df_final['listing_type_rule'].value_counts(dropna=False))

listing_type_rule
None                              5099
spare_screen_detected                5
wanted_request_detected              2
feature_phone_keyword_detected       2
tablet_keyword_detected              1
Name: count, dtype: int64


In [46]:
df_final.loc[
    df_final['listing_type'] == 'unclassified',
    'title'
].sample(20, random_state=42)

998                                                بطارية جوجل بيكسل ٨ برو
1297                                                   موتورولا موتو جى 85
850                                                             فيفو اس 19
3420                                                     آبل - آيفون 8 بلس
79                                                          آبل - آيفون 17
4714                                                       إنفنيكس هوت 11s
3309                                               آبل - آيفون 14 برو ماكس
2904                                                        آبل - آيفون 13
3019                                                      vivo v50 lite 5g
2040                 Samsung S25 Ultra 512 giga silver blue local warranty
766                    Apple - iPhone 16 Pro Max 256 ( zero ) battery 100%
33                                                          آبل - آيفون 13
1719                                                        اوبو رينو 8 تي
3597                     

In [47]:
battery_pattern = r'^\s*بطاري[ةه]'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(battery_pattern, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'parts'
df_final.loc[mask, 'listing_type_rule'] = 'spare_battery_detected'

print(df_final['listing_type'].value_counts())

listing_type
unclassified     5098
parts               6
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64


In [48]:
phone_pattern = r'iphone|samsung|galaxy|oppo|vivo|xiaomi|redmi|infinix|realme|motorola|honor|huawei|pixel|موتورولا|ايفون|سامسونج|شاومي|اوبو|فيفو|انفنيكس|هواوي'

phone_candidates = df_final[
    (df_final['listing_type'] == 'unclassified') &
    (df_final['title'].str.contains(phone_pattern, case=False, na=False))
]

print("Phone Candidates:", len(phone_candidates))
phone_candidates['title'].head(20)

Phone Candidates: 3144


0                                                            ايفون 13 iPhone
1                                                             Honor Magic V5
2                                  OPPO Find X9 Pro ضمان محلي للبيع او البدل
3                                             Honor Robot Phone 512G 12G ram
4                                             شاومي 17 الترا xiaomi 17 ultra
5      للبيع iPhone 15 Pro 1TB_ايفون 15 برو_زيرو بسعر لقطة لسرعه البيع ممتاز
6     ايفون 11 بحاله الزيرو (iphone11) - سعر لقطه للجادين للسرعه البيع 64 gb
8                                                         Huawei nova 15 max
9            ايفون 16e جديد وارد الامارات مدفوع ضريبة | iPhone 16e Phone New
10                        Oppo Find X5 5G dual sim 256/8G Black بحالة ممتازة
11               Samsung S24 Ultra زيرو 256GB تيتانيوم مدفوع الضريبة بالضمان
13              Honor Magic V5 dual sim 512/16G White Gold جديد بضمان الوكيل
14                                                         Samsung S25 Ultra

In [49]:
evaluation_df['phone_rule_test'] = evaluation_df['title_manual'].str.contains(
    phone_pattern,
    case=False,
    na=False,
    regex=True
)

print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['phone_rule_test']
))

phone_rule_test  False  True 
listing_type                 
feature_phone        1      0
other                3      0
parts                3      1
phone               62    127
tablet               1      0
wanted               2      0


In [50]:
evaluation_df.loc[
    (evaluation_df['listing_type'] == 'parts') &
    evaluation_df['phone_rule_test'],
    'title_manual'
]

119    شاشة samsung a34 خلع اصلية
Name: title_manual, dtype: str

In [51]:
phone_pattern = r'iphone|samsung|galaxy|oppo|vivo|xiaomi|redmi|infinix|realme|motorola|honor|huawei|pixel|موتورولا|ايفون|سامسونج|شاومي|اوبو|فيفو|انفنيكس|هواوي'

parts_exclusion = r'^\s*(?:شاش[ةه]|بطاري[ةه])'

evaluation_df['phone_rule_test'] = (
    evaluation_df['title_manual'].str.contains(
        phone_pattern, case=False, na=False, regex=True
    ) &
    ~evaluation_df['title_manual'].str.contains(
        parts_exclusion, na=False, regex=True
    )
)

print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['phone_rule_test']
))

phone_rule_test  False  True 
listing_type                 
feature_phone        1      0
other                3      0
parts                4      0
phone               62    127
tablet               1      0
wanted               2      0


In [52]:
parts_exclusion = r'^\s*(?:شاش[ةه]|بطاري[ةه])'

mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(phone_pattern, case=False, na=False, regex=True) &
    ~df_final['title'].str.contains(parts_exclusion, na=False, regex=True)
)

df_final.loc[mask, 'listing_type'] = 'phone'
df_final.loc[mask, 'listing_type_rule'] = 'phone_brand_detected'

print(df_final['listing_type'].value_counts())

listing_type
phone            3144
unclassified     1954
parts               6
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64


In [53]:
accessory_pattern = r'جراب|شاحن|سماعة|سماعات|كابل|باور بانك|power bank|headphone|earbuds|charger|cover|case'

accessory_candidates = df_final[
    (df_final['listing_type'] == 'unclassified') &
    (df_final['title'].str.contains(accessory_pattern, case=False, na=False, regex=True))
]

print("Accessory Candidates:", len(accessory_candidates))
accessory_candidates['title'].head(20)

Accessory Candidates: 4


723     iqoo neo 9s pro plus snapdragon 8 gen3 12+12 gb ram 256 memory 5 cover
2350           ريلميc53مساحه128بلعلبه وكابل شاحن حاله نضيفه جدا السعر5500نهائي
2710      آبل - آيفون 12 ميني بدون ضريبه خط واحد معاه شاحنه الاصلى بدون كرتونه
3458                                         هونر إكس 6 سي معاه العلبه والشاحن
Name: title, dtype: str

In [54]:
df_final.loc[
    df_final['listing_type'] == 'unclassified',
    'title'
].sample(20, random_state=42)

4268                            إنفنيكس نوت 11
4563                            آبل - آيفون 12
2574                   آبل - آيفون 17 برو ماكس
2799                            آبل - آيفون 13
823                             آبل - آيفون 17
1181                             ريلمى 14 برو+
3846                       آبل - آيفون 12 ميني
1618                            آبل - آيفون 12
2882                   آبل - آيفون 15 برو ماكس
820                                      اي 51
999                               ريلمى 15 5 چ
4258                         إنفنيكس موديل اخر
3722                            آبل - آيفون 12
2221                            إنفنيكس هوت 50
2935                        آبل - آيفون 11 برو
3560                                هونر x 9 c
236                         تلفون ريدمي نوت 14
1709                            آبل - آيفون 12
1194                                 موتو E32s
410     موبايل شيك استعمال ٣ سنين بس قابل للطي
Name: title, dtype: str

In [55]:
phone_pattern = (
    r'iphone|apple|samsung|galaxy|oppo|vivo|xiaomi|redmi|'
    r'infinix|realme|motorola|honor|huawei|pixel|'
    r'موتورولا|ايفون|آيفون|آبل|سامسونج|شاومي|ريدمي|'
    r'اوبو|أوبو|فيفو|انفنيكس|إنفنيكس|ريلمي|ريلمى|'
    r'هونر|هواوي|موتو|تلفون|موبايل'
)

In [56]:
evaluation_df['phone_rule_test'] = (
    evaluation_df['title_manual'].str.contains(
        phone_pattern, case=False, na=False, regex=True
    ) &
    ~evaluation_df['title_manual'].str.contains(
        parts_exclusion, na=False, regex=True
    )
)

print(pd.crosstab(
    evaluation_df['listing_type'],
    evaluation_df['phone_rule_test']
))

phone_rule_test  False  True 
listing_type                 
feature_phone        1      0
other                3      0
parts                4      0
phone                7    182
tablet               1      0
wanted               1      1


In [57]:
mask = (
    df_final['listing_type'].eq('unclassified') &
    df_final['title'].str.contains(
        phone_pattern, case=False, na=False, regex=True
    ) &
    ~df_final['title'].str.contains(
        parts_exclusion, na=False, regex=True
    )
)

df_final.loc[mask, 'listing_type'] = 'phone'
df_final.loc[mask, 'listing_type_rule'] = 'phone_brand_detected'

print(df_final['listing_type'].value_counts())

listing_type
phone            4732
unclassified      366
parts               6
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64


In [58]:
print(df_final['listing_type_rule'].value_counts(dropna=False))

listing_type_rule
phone_brand_detected              4732
None                               366
spare_screen_detected                5
wanted_request_detected              2
feature_phone_keyword_detected       2
spare_battery_detected               1
tablet_keyword_detected              1
Name: count, dtype: int64


In [59]:
df_final.to_csv('train_clean_5_1.csv', index=False)

Apply listing type rules on validation data

In [60]:
# Load Original Validation Dataset
val_df = pd.read_csv("../data/processed/price_review_task4/val_scored_4_1.csv")

# Create a separate copy
val_final = val_df.copy()

# Initialize listing type columns
val_final['listing_type'] = 'unclassified'
val_final['listing_type_rule'] = None

# 1. Wanted listings
wanted_pattern = r'^\s*محتاج\s+(?:هاتف|تلفون|موبايل)'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(wanted_pattern, na=False, regex=True)
)
val_final.loc[mask, 'listing_type'] = 'wanted'
val_final.loc[mask, 'listing_type_rule'] = 'wanted_request_detected'

# 2. Spare screens
parts_pattern = r'^\s*شاش[ةه]'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(parts_pattern, na=False, regex=True)
)
val_final.loc[mask, 'listing_type'] = 'parts'
val_final.loc[mask, 'listing_type_rule'] = 'spare_screen_detected'

# 3. Tablets
tablet_pattern = r'\b(?:matepad|ipad|galaxy tab|lenovo tab|tablet|تابلت)\b'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(tablet_pattern, case=False, na=False, regex=True)
)
val_final.loc[mask, 'listing_type'] = 'tablet'
val_final.loc[mask, 'listing_type_rule'] = 'tablet_keyword_detected'

# 4. Feature phones
feature_phone_pattern = r'اريكسون|زرار|feature phone|نوكيا'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(feature_phone_pattern, case=False, na=False, regex=True)
)
val_final.loc[mask, 'listing_type'] = 'feature_phone'
val_final.loc[mask, 'listing_type_rule'] = 'feature_phone_keyword_detected'

# Exclude Nokia from feature_phone
nokia_mask = (
    val_final['listing_type'].eq('feature_phone') &
    val_final['title'].str.contains('نوكيا', na=False)
)
val_final.loc[nokia_mask, 'listing_type'] = 'unclassified'
val_final.loc[nokia_mask, 'listing_type_rule'] = None

# 5. Spare batteries
battery_pattern = r'^\s*بطاري[ةه]'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(battery_pattern, na=False, regex=True)
)
val_final.loc[mask, 'listing_type'] = 'parts'
val_final.loc[mask, 'listing_type_rule'] = 'spare_battery_detected'

# 6. Phone listings
phone_pattern = (
    r'iphone|apple|samsung|galaxy|oppo|vivo|xiaomi|redmi|'
    r'infinix|realme|motorola|honor|huawei|pixel|'
    r'موتورولا|ايفون|آيفون|آبل|سامسونج|شاومي|ريدمي|'
    r'اوبو|أوبو|فيفو|انفنيكس|إنفنيكس|ريلمي|ريلمى|'
    r'هونر|هواوي|موتو|تلفون|موبايل'
)

parts_exclusion = r'^\s*(?:شاش[ةه]|بطاري[ةه])'

mask = (
    val_final['listing_type'].eq('unclassified') &
    val_final['title'].str.contains(phone_pattern, case=False, na=False, regex=True) &
    ~val_final['title'].str.contains(parts_exclusion, na=False, regex=True)
)

val_final.loc[mask, 'listing_type'] = 'phone'
val_final.loc[mask, 'listing_type_rule'] = 'phone_brand_detected'

# Results
print("Validation Shape:", val_final.shape)

print("\nListing Type Distribution:")
print(val_final['listing_type'].value_counts(dropna=False))

print("\nApplied Rules:")
print(val_final['listing_type_rule'].value_counts(dropna=False))

Validation Shape: (1252, 66)

Listing Type Distribution:
listing_type
phone           1155
unclassified      97
Name: count, dtype: int64

Applied Rules:
listing_type_rule
phone_brand_detected    1155
None                      97
Name: count, dtype: int64


In [61]:
unclassified_titles = val_final.loc[
    val_final['listing_type'] == 'unclassified',
    'title'
]

print("Unclassified Count:", len(unclassified_titles))

display(
    unclassified_titles.sample(
        min(20, len(unclassified_titles)),
        random_state=42
    )
)

Unclassified Count: 97


817                                                               وان بلس 15
601                          زيرو ما فيه خدش ومعاه ملحقاته الكرتونة و الشاحن
1166                                                  i phone 15 pro max 512
347                                                           للبيع بوكو C85
1003                                                           Sony Ericsson
1028                                           Asus TP550 LP - شاشه ٣٦٠ درجه
842                                                            جلاكسي نوت 20
606                                                                   الجيزه
221                                                                مدينة نصر
19                                                                أخرىسفرة ح
487                                                      محتاج موبيل ب ٣٥٠٠ج
929                                                          Sony Xperia 1 V
675                                                      انفنكس هوت 50 للبدل

In [62]:
val_final.to_csv("validation_with_listing_type.csv", index=False)